In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import boxcox
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.nn.utils.parametrizations import weight_norm
from torch.utils.data import TensorDataset, DataLoader, Dataset

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

In [ ]:
class OutputCrop1d(nn.Module):
    def __init__(self, crop_size: int):
        super().__init__()
        self.crop_size = crop_size

    def forward(self, x: torch.Tensor):
        return x[:, :, :-self.crop_size].contiguous()


class TemporalConvUnit(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int,
        padding: int,
        dilation: int,
        stride: int = 1,
        dropout: float = 0.2,
        name: str | None = None
    ):
        super().__init__()
        self.name = name
        
        # Weight normalisation: https://arxiv.org/abs/1602.07868
        self.conv = weight_norm(
            nn.Conv1d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=kernel_size,
                padding=padding,
                dilation=dilation,
                stride=stride,
            )
        )
        self.conv.weight.data.normal_(0, 0.01)
        self.crop = OutputCrop1d(padding)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.net = nn.Sequential(self.conv, self.crop, self.relu, self.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)
    

class TemporalConvBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int,
        padding: int,
        dilation: int,
        stride: int = 1,
        dropout: float = 0.2,
    ):
        """
        :param in_channels: Number of input channels.
            Corresponds to the number of features at each timestep in the input series.
        :param out_channels: Number of output channels.
            Corresponds to the number of features at each timestep in the output series.
        :param kernel_size: Number of weights per filter.
        :param padding: Size of padding to apply to both sides of the input
        """
        super().__init__()
        
        self.unit1 = TemporalConvUnit(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=padding,
            stride=stride,
            dropout=dropout,
        )
        self.unit2 = TemporalConvUnit(
            in_channels=out_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=padding,
            stride=stride,
            dropout=dropout,
        )
        self.net = nn.Sequential(self.unit1, self.unit2)

        # Residual connection
        if in_channels != out_channels:
            self.conv = nn.Conv1d(in_channels, out_channels, kernel_size=1)
            self.conv.weight.data.normal_(0, 0.01)
        else:
            self.conv = None
        
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.net(x)
        res = x if self.conv is None else self.conv(x)
        return self.relu(out + res)
    

class TemporalConvNetDecoder(nn.Module):
    def __init__(self, in_features: int, horizon: int = 1):
        super().__init__()
        """
        Linear decoder that maps the final hidden representation from the TCN 
        into the target forecasting horizon.

        :param in_features: Number of input features (channels) from the final TCN layer. 
            This corresponds to the number of learned feature maps at the last timestep.

        :param horizon: Number of future timesteps to predict. 
            The decoder outputs one value per step in the forecast horizon.
        """
        self.linear = nn.Linear(in_features=in_features, out_features=horizon)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x)


class TemporalConvNet(nn.Module):
    def __init__(
        self,
        in_features: int,
        out_features: list[int],
        horizon: int, 
        kernel_size: int = 2,
        dropout: float = 0.2
    ):
        """
        :param in_features: Number of input features at each timestep in the time series.
            This corresponds to the number of input channels to the first convolutional layer.
        
        :param out_features: List specifying the number of output feature maps (channels) 
            for each temporal convolutional block in the network.
            For example, [16, 32, 64] creates three stacked convolutional blocks with
            16, 32, and 64 output channels, respectively.
        
        :param horizon: Number of future timesteps to predict i.e. the forecasting horizon
        
        :param kernel_size: Size of the temporal convolution kernel.
            Controls the receptive field of each convolutional layer.
        
        :param dropout: Dropout probability applied after each convolutional layer.
        """
        super().__init__()
        
        layers = []
        n_layers = len(out_features)
        for i in range(n_layers):
            in_channels = in_features if i == 0 else out_features[i - 1]
            out_channels = out_features[i]
            dilation_size = 2 ** i
            conv_block = TemporalConvBlock(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=kernel_size,
                dilation=dilation_size,
                padding=(kernel_size - 1) * dilation_size,
                dropout=dropout
            )
            layers.append(conv_block)
                

        self.encoder = nn.Sequential(*layers)
        self.decoder = TemporalConvNetDecoder(out_features[-1], horizon)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        encoded = self.encoder(x)
        # Only select the final timestep feature maps
        # for forecasting
        return self.decoder(encoded[:, :, -1])

In [ ]:
batch_size = 32
in_seq_length = 35
in_features = 1

out_features = [16, 32, 64]
out_seq_length = 23

model = TemporalConvNet(
    in_features=in_features,
    out_features=out_features,
    horizon=out_seq_length,
)

in_ = torch.randn(batch_size, in_seq_length, in_features)
in_ = in_.permute(0, 2, 1)

out_ = model(in_)

In [ ]:
n_timesteps = 1000
period = 24
timesteps = np.arange(n_timesteps)
timeseries = np.sin(2 * np.pi * timesteps / period)

plt.plot(timesteps, timeseries)

In [ ]:
def prepare_single_horizon_train_dataset(
    timeseries: np.ndarray,
    in_seq_length: int,
    out_seq_length: int,
    target_col_index: int = 0,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Prepares input–output pairs for single-horizon time series forecasting.

    This function constructs overlapping input sequences (features) and corresponding
    output sequences (labels) from a continuous time series. Each input window of 
    length `in_seq_length` is paired with the immediately following `out_seq_length` 
    values as the forecast target.

    :param timeseries: The full time series array of shape (T, n) where n is the number of input features.
        n = 1 for univariate timeseries forecasting with only the timeseries itself as feature inputs.
    :param in_seq_length: Length of each input sequence (number of past timesteps used as context).
    :param out_seq_length: Number of future timesteps to predict (forecast horizon).
    :param target_col_index: Column index of the target variable. Defaults to 0.
    """

    if timeseries.ndim < 2:
        timeseries = timeseries.reshape(-1, 1)
    elif timeseries.ndim > 2:
        raise ValueError("Expecting input array with at most two dimensions.")

    features, labels = [], []
    max_ts_index = len(timeseries) - in_seq_length - out_seq_length + 1
    n_in_features = timeseries.shape[-1]
    for i in tqdm(range(max_ts_index)):
        feat_start, feat_end = i, i + in_seq_length
        feat_seq = timeseries[feat_start: feat_end]
        features.append(feat_seq)
        
        labels_start = i + in_seq_length
        labels_end = labels_start + out_seq_length
        labels_seq = timeseries[labels_start: labels_end, [target_col_index]]
        labels.append(labels_seq)

    features_ts = torch.tensor(np.array(features), dtype=torch.float)
    features_ts = features_ts.view(-1, in_seq_length, n_in_features)

    labels_ts = torch.tensor(np.array(labels), dtype=torch.float32)
    labels_ts = labels_ts.view(-1, out_seq_length, 1)

    return features_ts, labels_ts


def prepare_multi_horizon_train_dataset(
    timeseries: np.ndarray,
    in_seq_length: int,
    out_seq_length: int,
    in_features: int = 1,
) -> tuple[np.ndarray, np.ndarray]:
    
    features, labels = [], []
    max_ts_index = len(timeseries) - in_seq_length - out_seq_length + 1
    for i in range(max_ts_index):
        feat_start, feat_end = i, i + in_seq_length
        feat_seq = timeseries[feat_start: feat_end]
        features.append(feat_seq)
        for j in range(in_seq_length):
            labels_start = i + j + 1
            labels_end = labels_start + out_seq_length
            labels_seq = timeseries[labels_start: labels_end]
            labels.append(labels_seq)
    
    features_ts = torch.tensor(np.array(features), dtype=torch.float)
    features_ts = features_ts.view(-1, in_seq_length, in_features)

    labels_ts = torch.tensor(np.array(labels), dtype=torch.float32)
    labels_ts = labels_ts.view(-1, in_seq_length, out_seq_length)

    return features_ts, labels_ts

In [ ]:
in_seq_length = 2 * period
out_seq_length = period
X, y = prepare_single_horizon_train_dataset(
    timeseries=timeseries,
    in_seq_length=in_seq_length,
    out_seq_length=out_seq_length,
)

train_ds = TensorDataset(X, y)
train_dl = DataLoader(train_ds, batch_size=32)

In [ ]:
model = TemporalConvNet(
    in_features=1,
    out_features=[16, 32, 64],
    horizon=out_seq_length,
    kernel_size=2,
    dropout=0.2
)
model = model.to(DEVICE)

loss_fn = nn.MSELoss()
optimizer = AdamW(model.parameters(), lr=1e-03)

model.train()
epoch_loss, batch_loss = [], []
n_epochs = 50
epoch_pgbar = tqdm(range(n_epochs))
for epoch in epoch_pgbar:
    for batch_X, batch_y in train_dl:
        batch_X = batch_X.to(DEVICE)
        batch_y = batch_y.to(DEVICE)
        
        optimizer.zero_grad()
        
        # Typically sequence modelling have (batch_size, in_seq_length, num_input_fetures)
        # But CNNs work with (batch_size, num_input_features, in_seq_length) convention
        batch_X = batch_X.permute(0, 2, 1)
        y_hat = model(batch_X)

        # Output dimension is (batch_size, out_seq_length)
        y_hat = y_hat.unsqueeze(-1)
        loss = loss_fn(batch_y, y_hat)
        loss.backward()
        optimizer.step()
        
        loss_detach = float(loss.detach())
        batch_loss.append(loss_detach)
    
    epoch_loss.append(loss_detach)
    epoch_pgbar.set_description(f"Epoch [{epoch + 1} / {n_epochs}] - Loss = {loss_detach:.3f}")

In [ ]:
plt.plot(batch_loss)

In [ ]:
plt.plot(epoch_loss)

In [ ]:
# Forecast
X_test = timeseries[-in_seq_length - out_seq_length: - out_seq_length]
X_test = torch.tensor(X_test, dtype=torch.float).view(-1, in_seq_length, 1)

y_test = timeseries[-out_seq_length: ]

model.eval()
y_pred = model(X_test.permute(0, 2, 1))

plt.plot(y_pred.squeeze().detach().numpy())
plt.plot(y_test)

## UCI Dataset

In [ ]:
from datetime import datetime, timedelta
import numpy as np
import polars as pl


# from preprocess.explore import get_min_max_timestamps_by_client
# from preprocess.constants import (
#     UCI_CLIENTS_TO_DROP,
#     UCI_CLIENTS_TO_FILTER,
#     UCI_CLIENTS_TO_INTERPOLATE
# )
# from preprocess.filter import drop_client_timeseries, filter_client_timeseries
# from preprocess.impute import interpolate_client_timeseries
# from validate.constants import UCI_CLIENT_SITES_TO_VALIDATE

In [ ]:
def get_min_max_timestamps_by_client(uci_df: pl.DataFrame) -> pl.DataFrame:
    min_max_ts = (
        uci_df.unpivot(
            on=[c for c in uci_df.columns if c != "timestamp"],
            index="timestamp",
            variable_name="client",
        )
        .filter(pl.col("value") > 0)
        .group_by("client", maintain_order=True)
        .agg(
            min_timestamp=pl.col("timestamp").min(),
            max_timestamp=pl.col("timestamp").max(),
        )
    )
    return min_max_ts


def drop_client_timeseries(uci_df: pl.DataFrame, client_name: str) -> pl.DataFrame:
    return uci_df.select(pl.all().exclude(client_name))


def filter_client_timeseries(
    uci_df: pl.DataFrame,
    client_name: str,
    start_ts: datetime,
    end_ts: datetime,
) -> pl.DataFrame:

    client_df = uci_df.select(pl.col("timestamp"), pl.col(client_name)).filter(
        pl.col("timestamp").is_between(start_ts, end_ts, closed="both")
    )

    # Merge back onto main df
    uci_df = (
        uci_df.select(pl.all().exclude(client_name))
        .join(client_df, on="timestamp", how="left")
        .select(pl.all().exclude(client_name), pl.col(client_name).fill_null(0.0))
    )

    # Sort columns
    uci_df = uci_df.select(
        pl.col("timestamp"),
        *[pl.col(c) for c in sorted([c for c in uci_df.columns if c != "timestamp"])]
    )

    return uci_df


def interpolate_client_timeseries(
    uci_df: pl.DataFrame,
    client_name: str,
    start_ts: datetime,
    end_ts: datetime,
    interval: str = "15m",
) -> pl.DataFrame:
    """
    Filter a client timeseries to between start_ts and end_ts, and interpolate
    """

    # Get all non-zero observations between start_ts and end_ts
    client_df = uci_df.select(pl.col("timestamp"), pl.col(client_name)).filter(
        pl.col("timestamp").is_between(start_ts, end_ts, closed="both"),
        pl.col(client_name) > 0,
    )

    # Construct timeseries of expected timestamps between start_ts and end_ts
    expected_ts = pl.datetime_range(
        start=start_ts,
        end=end_ts,
        interval=interval,
        closed="both",
        eager=True,
    )

    # Merge and interpoalte
    client_df = (
        expected_ts.to_frame(name="timestamp")
        .join(client_df, on="timestamp", how="left")
        .select(pl.col("timestamp"), pl.col(client_name).interpolate())
    )

    # Merge back onto original uci_df
    uci_df = (
        uci_df.select(pl.all().exclude(client_name))
        .join(client_df, on="timestamp", how="left")
        .select(pl.all().exclude(client_name), pl.col(client_name).fill_null(0.0))
    )

    # Sort columns
    uci_df = uci_df.select(
        pl.col("timestamp"),
        *[pl.col(c) for c in sorted([c for c in uci_df.columns if c != "timestamp"])],
    )

    return uci_df


In [ ]:
# constants imports
from datetime import datetime

UCI_CLIENTS_TO_DROP: list[str] = [
    "MT_288",
    "MT_066",
    "MT_127",
]


UCI_CLIENTS_TO_INTERPOLATE: list[str] = [
    "MT_002",
    "MT_094",
    "MT_095",
    "MT_097",
    "MT_098",
    "MT_099",
    "MT_100",
    "MT_102",
    "MT_104",
    "MT_105",
    "MT_096",
    "MT_092",
    "MT_223",
    "MT_030",
    "MT_024",
    "MT_091",
    "MT_093",
    "MT_101",
    "MT_124",
    "MT_228",
    "MT_230",
    "MT_165",
    "MT_041",
    "MT_241",
    "MT_187",
    "MT_204",
    "MT_212",
    "MT_222",
    "MT_243",
    "MT_247",
    "MT_272",
    "MT_033",
    "MT_103",
    "MT_012",
    "MT_210",
    "MT_215",
    "MT_220",
    "MT_274",
    "MT_328",
    "MT_186",
    "MT_005",
    "MT_006",
    "MT_011",
    "MT_018",
    "MT_021",
    "MT_025",
    "MT_026",
    "MT_038",
    "MT_040",
    "MT_044",
    "MT_045",
    "MT_046",
    "MT_050",
    "MT_051",
    "MT_053",
    "MT_054",
    "MT_059",
    "MT_069",
    "MT_073",
    "MT_078",
    "MT_085",
    "MT_087",
    "MT_089",
    "MT_125",
    "MT_126",
    "MT_128",
    "MT_135",
    "MT_137",
    "MT_139",
    "MT_140",
    "MT_141",
    "MT_142",
    "MT_147",
    "MT_148",
    "MT_339",
    "MT_200",
    "MT_206",
    "MT_221",
    "MT_291",
    "MT_152",
    "MT_039",
    "MT_106",
    "MT_107",
    "MT_108",
    "MT_110",
    "MT_111",
    "MT_113",
    "MT_117",
    "MT_120",
    "MT_121",
    "MT_122",
    "MT_032",
    "MT_189",
    "MT_202",
    "MT_217",
    "MT_004",
    "MT_007",
    "MT_008",
    "MT_016",
    "MT_020",
    "MT_027",
    "MT_029",
    "MT_035",
    "MT_036",
    "MT_037",
    "MT_042",
    "MT_043",
    "MT_047",
    "MT_056",
    "MT_058",
    "MT_060",
    "MT_061",
    "MT_062",
    "MT_063",
    "MT_068",
    "MT_070",
    "MT_071",
    "MT_072",
    "MT_074",
    "MT_076",
    "MT_077",
    "MT_079",
    "MT_082",
    "MT_083",
    "MT_084",
    "MT_086",
    "MT_088",
    "MT_138",
    "MT_155",
    "MT_369",
    "MT_161",
    "MT_191",
    "MT_227",
    "MT_314",
    "MT_325",
    "MT_116",
    "MT_010",
    "MT_013",
    "MT_019",
    "MT_022",
    "MT_028",
    "MT_031",
    "MT_034",
    "MT_048",
    "MT_049",
    "MT_052",
    "MT_055",
    "MT_065",
    "MT_067",
    "MT_081",
    "MT_118",
    "MT_150",
    "MT_199",
    "MT_009",
    "MT_075",
    "MT_145",
    "MT_195",
    "MT_197",
    "MT_163",
    "MT_190",
    "MT_119",
    "MT_343",
    "MT_358",
    "MT_365",
    "MT_368",
    "MT_196",
    "MT_208",
    "MT_361",
    "MT_211",
    "MT_109",
    "MT_090",
    "MT_194",
    "MT_198",
    "MT_214",
    "MT_218",
    "MT_273",
    "MT_144",
    "MT_153",
    "MT_201",
    "MT_216",
    "MT_353",
    "MT_193",
    "MT_207",
    "MT_149",
    "MT_342",
    "MT_164",
    "MT_351",
    "MT_356",
    "MT_114",
    "MT_363",
    "MT_162",
    "MT_352",
    "MT_213",
    "MT_350",
    "MT_344",
    "MT_146",
    "MT_154",
    "MT_014",
    "MT_080",
    "MT_226",
    "MT_354",
    "MT_192",
    "MT_331",
    "MT_359",
    "MT_157",
    "MT_322",
    "MT_017",
    "MT_360",
    "MT_364",
    "MT_115",
    "MT_357",
    "MT_219",
    "MT_174",
    "MT_175",
    "MT_188",
    "MT_236",
    "MT_246",
    "MT_279",
    "MT_282",
    "MT_313",
    "MT_367",
    "MT_362",
    "MT_345",
    "MT_203",
    "MT_267",
    "MT_366",
    "MT_205",
    "MT_209",
    "MT_229",
    "MT_023",
    "MT_136",
    "MT_335",
    "MT_338",
    "MT_349",
    "MT_064",
    "MT_341",
    "MT_151",
    "MT_333",
    "MT_336",
    "MT_112",
    "MT_346",
    "MT_143",
    "MT_334",
    "MT_355",
    "MT_340",
    "MT_123",
    "MT_370",
    "MT_332",
    "MT_179",
    "MT_337",
    "MT_181",
    "MT_003",
    "MT_178",
    "MT_057",
]


UCI_CLIENTS_TO_FILTER: list[tuple[str, tuple[datetime, datetime]]] = [
    (
        "MT_015",
        (datetime(2013, 11, 19), datetime(2015, 1, 1)),
    ),
]


UCI_CLIENT_SITES_TO_VALIDATE: list[str] = [
    "MT_156",
    "MT_162",
    "MT_189",
    "MT_190",
    "MT_191",
    "MT_205",
    "MT_212",
    "MT_217",
    "MT_240",
    "MT_251",
    "MT_261",
    "MT_262",
    "MT_263",
    "MT_267",
    "MT_280",
    "MT_297",
    "MT_299",
    "MT_307",
    "MT_321",
    "MT_329",
]

In [ ]:
FREQUENCY_MINUTES = 15
# UCI_DATA_PATH = "./data/LD2011_2014.txt"
UCI_DATA_PATH = "/kaggle/input/uci-electricity-load-2011-2014/LD2011_2014.txt"

In [ ]:
# Load data as polars dataframe
UCI_DF = pl.read_csv(
    UCI_DATA_PATH,
    has_header=True,
    separator=";",
    decimal_comma=True,
    try_parse_dates=True,
    infer_schema_length=1_000_000
)

# First column should be timestamp column
UCI_DF = UCI_DF.rename({UCI_DF.columns[0]: "timestamp"}).sort(by="timestamp")

In [ ]:
# Data processing
for client in UCI_CLIENTS_TO_DROP:
    UCI_DF = drop_client_timeseries(uci_df=UCI_DF, client_name=client)


# Filter client timeseries
for client_name, (start_ts, end_ts) in UCI_CLIENTS_TO_FILTER:
    UCI_DF = filter_client_timeseries(
        uci_df=UCI_DF,
        client_name=client_name,
        start_ts=start_ts,
        end_ts=end_ts
    )


# Interpolate client timeseries
min_max_timestamp_by_client = get_min_max_timestamps_by_client(UCI_DF)
for client_name in UCI_CLIENTS_TO_INTERPOLATE:
    # Get min / max timestamps for this client
    client_min_max_ts = min_max_timestamp_by_client.filter(pl.col("client") == client_name)
    [client_min_ts] = client_min_max_ts["min_timestamp"].to_list()
    [client_max_ts] = client_min_max_ts["max_timestamp"].to_list()

    UCI_DF = interpolate_client_timeseries(
        uci_df=UCI_DF,
        client_name=client_name,
        start_ts=client_min_ts,
        end_ts=client_max_ts,
        interval=f"{FREQUENCY_MINUTES}m"
    )

In [ ]:
VALIDATION_START = datetime(2014, 12, 1)
VALIDATION_WINDOW = timedelta(days=2)
N_FOLDS = 10

In [ ]:
# Unpivot UCI DF

long_sample_clients_demand_table = (
    UCI_DF
    .unpivot(
        on=[c for c in UCI_DF.columns if c != "timestamp"],
        index="timestamp",
        variable_name="client",
        value_name="demand"
    )
    .join(
        other=min_max_timestamp_by_client,
        on="client",
        how="left",
    )
    .with_columns(
        in_range=pl.col("timestamp").is_between(pl.col("min_timestamp"), pl.col("max_timestamp")),
        log1p_demand=pl.col("demand").log1p(),
    )
    .filter(
        pl.col("in_range"),
        pl.col("client").is_in(UCI_CLIENT_SITES_TO_VALIDATE)
    )
)

In [ ]:
for client in UCI_CLIENT_SITES_TO_VALIDATE:
    for i in range(N_FOLDS):
        val_start = VALIDATION_START + VALIDATION_WINDOW * i
        val_end = val_start + VALIDATION_WINDOW
        client_train_df = (
            long_sample_clients_demand_table
            .filter(pl.col("client") == client, pl.col("timestamp").lt(val_start))
            .select(pl.col("timestamp"), pl.col("demand"))
            .sort(by="timestamp")
        )
        client_val_df = (
            long_sample_clients_demand_table
            .filter(pl.col("client") == client, pl.col("timestamp").is_between(val_start, val_end, closed="left"))
            .select(pl.col("timestamp"), pl.col("demand"))
            .sort(by="timestamp")
        )

        # Feature engineering
        mean, std = client_train_df["demand"].mean(), client_train_df["demand"].std()
        client_train_df = (
            client_train_df
            .with_columns(
                demand_scaled=(pl.col("demand") - mean) / std,
                
                hour_of_day=pl.col("timestamp").dt.hour(),
                sin_hour_of_day=(pl.col("timestamp").dt.hour() * 2 * np.pi / 24).sin(),
                cos_hour_of_day=(pl.col("timestamp").dt.hour() * 2 * np.pi / 24).cos(),

                day_of_week=pl.col("timestamp").dt.weekday(),
                sin_day_of_week=(pl.col("timestamp").dt.weekday() * 2 * np.pi / 7).sin(),
                cos_day_of_week=(pl.col("timestamp").dt.weekday() * 2 * np.pi / 7).cos(),
                is_weekend=(pl.col("timestamp").dt.weekday() >= 6).cast(pl.Float64),
                
                month_of_year=pl.col("timestamp").dt.month(),
                sin_month_of_year=(pl.col("timestamp").dt.month() * 2 * np.pi / 12).sin(),
                cos_month_of_year=(pl.col("timestamp").dt.month() * 2 * np.pi / 12).cos(),
            )
        )
        
        break

    break

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, timeseries: np.ndarray, in_seq_length: int, out_seq_length: int, target_col_idx: int = 0):
        super().__init__()
        if timeseries.ndim < 2:
            timeseries = timeseries.reshape(-1, 1)
        elif timeseries.ndim > 2:
            raise ValueError("Expecting input array with at most two dimensions.")
        
        self.timeseries = torch.tensor(timeseries, dtype=torch.float32)
        self.in_seq_length = in_seq_length
        self.out_seq_length = out_seq_length
        self.target_col_index = target_col_idx

    def __len__(self):
        return len(self.timeseries) - self.in_seq_length - self.out_seq_length + 1
    
    def __getitem__(self, idx) -> tuple[torch.Tensor, torch.Tensor]:
        x_start, x_end = int(idx), int(idx + self.in_seq_length)
        x = self.timeseries[x_start: x_end]
        
        y_start, y_end = int(x_end), int(x_end + self.out_seq_length)
        y = self.timeseries[y_start: y_end, [self.target_col_index]]
        return x, y

In [ ]:
FEATURES = [
    "demand_scaled", 
    "sin_hour_of_day",
    "cos_hour_of_day",
    "sin_day_of_week",
    "cos_day_of_week",
    "sin_month_of_year",
    "cos_month_of_year",
    "is_weekend"
]

in_seq_length = int(30 * 24 * 60 / FREQUENCY_MINUTES)  # use 30 days of input
out_seq_length = int(2 * 24 * 60 / FREQUENCY_MINUTES)  # predict 2 days of data

timeseries = client_train_df[FEATURES].to_numpy()
train_ds = TimeSeriesDataset(
    timeseries=timeseries,
    in_seq_length=in_seq_length,
    out_seq_length=out_seq_length,
    target_col_idx=0,
)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)

In [ ]:
model = TemporalConvNet(
    in_features=timeseries.shape[-1],
    out_features=[8, 16, 32, 64],
    horizon=out_seq_length,
    kernel_size=2,
    dropout=0.2
)
model = model.to(DEVICE)

loss_fn = nn.MSELoss()
optimizer = AdamW(model.parameters(), lr=1e-03)

model.train()
epoch_loss, batch_loss = [], []
n_epochs = 50
epoch_pgbar = tqdm(range(n_epochs))
for epoch in range(n_epochs):
    for batch_X, batch_y in tqdm(train_dl):
        
        batch_X = batch_X.to(DEVICE)
        batch_y = batch_y.to(DEVICE)

        optimizer.zero_grad()
        
        # Typically sequence modelling have (batch_size, in_seq_length, num_input_fetures)
        # But CNNs work with (batch_size, num_input_features, in_seq_length) convention
        batch_X = batch_X.permute(0, 2, 1)
        y_hat = model(batch_X)

        # Output dimension is (batch_size, out_seq_length)
        y_hat = y_hat.unsqueeze(-1)
        loss = loss_fn(batch_y, y_hat)
        loss.backward()
        optimizer.step()
        
        loss_detach = float(loss.detach())
        batch_loss.append(loss_detach)
    
    break
    
    epoch_loss.append(loss_detach)
    epoch_pgbar.set_description(f"Epoch [{epoch + 1} / {n_epochs}] - Loss = {loss_detach:.3f}")

In [ ]:
plt.plot(batch_loss)